In [98]:
import os
import io
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ==========================================
# 1. KONFIGURACE
# ==========================================
class Config:
    # sekvence
    window_size = 80

    # model
    embed_size = 64
    n_heads = 4
    n_layers = 3
    dropout = 0.3

    # trénink
    batch_size = 32
    lr = 2e-4
    epochs = 60
    weight_decay = 1e-4

    # tokenizace
    default_num_bins = 128
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = Config()

# ==========================================
# 2. VÝBĚR SLOUPCŮ PRO MODEL A
#    G1 + G2 + D1 + D2 -> G3(posun, zdvih)
# ==========================================

G1_FEATURES = [
    'posun [mm]_G1',
    'zdvih [mm]_G1',
]

G2_FEATURES = [
    'posun [mm]_G2',
    'zdvih [mm]_G2',
]

D1_FEATURES = [
    'Cant_D1',
    'Twist_D1',
    'LevelLeft_D1',
    'LevelRight_D1',
]

D2_FEATURES = [
    'Cant_D2',
    'Twist_D2',
    'LevelLeft_D2',
    'LevelRight_D2',
]

DELTA_FEATURES = [
    'delta_posun',
    'delta_zdvih',
]

ROLLING_FEATURES = [
    'posun_G2_roll5',
    'zdvih_G2_roll5',
]

ALL_INPUT_FEATURES = (
    G1_FEATURES +
    G2_FEATURES +
    D1_FEATURES +
    D2_FEATURES +
    DELTA_FEATURES +
    ROLLING_FEATURES
)

TARGETS = ['posun [mm]_G3',
           'zdvih [mm]_G3',
]

In [ ]:
# ==========================================
# 3. NAČTENÍ SOUBORU
# ==========================================
def load_track_data():

    try:
        from google.colab import files
        print("Běžíš v Colabu. Vyber soubor .xlsx:")
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]
        content = io.BytesIO(uploaded[fname])
    except Exception:
        import tkinter as tk
        from tkinter import filedialog

        print("Běžíš lokálně. Otevírám okno pro výběr souboru...")
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        fname = filedialog.askopenfilename(filetypes=[("Excel", "*.xlsx")])
        root.destroy()
        content = fname

    if not fname:
        raise ValueError("Nebyl vybrán žádný soubor.")

    print(f"Načítám data ze souboru: {fname}")

    g1 = pd.read_excel(content, sheet_name='G1_vs_D1', engine='openpyxl')
    g2 = pd.read_excel(content, sheet_name='G2_vs_D2', engine='openpyxl')
    g3 = pd.read_excel(content, sheet_name='G3_final', engine='openpyxl')

    return g1, g2, g3, fname


def validate_columns(df, required_cols, df_name):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"V {df_name} chybí sloupce: {missing}")


def basic_data_check(g1, g2, g3):
    validate_columns(g1, G1_FEATURES + D1_FEATURES, "G1_vs_D1")
    validate_columns(g2, G2_FEATURES + D2_FEATURES, "G2_vs_D2")
    validate_columns(g3, TARGETS, "G3_final")

    lengths = [len(g1), len(g2), len(g3)]
    if len(set(lengths)) != 1:
        raise ValueError(f"Listy nemají stejnou délku: {lengths}")

    print("\nKontrola dat OK")
    print(f"Počet řádků: {lengths[0]}")
    print(f"Počet vstupních feature: {len(ALL_INPUT_FEATURES)}")
    print(f"Targety: {TARGETS}")


# Načtení
df_g1, df_g2, df_g3, selected_file = load_track_data()
basic_data_check(df_g1, df_g2, df_g3)

In [100]:
# ==========================================
# 4. TOKENIZACE PO FEATURE
# ==========================================
class FeatureTokenizer:
    def __init__(self, num_bins=128):
        self.num_bins = num_bins
        self.feature_bins = {}
        self.feature_mins = {}
        self.feature_maxs = {}

    def fit_feature(self, name, values):
        values = np.asarray(values, dtype=np.float32)
        vmin = np.nanmin(values)
        vmax = np.nanmax(values)

        if math.isclose(vmin, vmax):
            vmax = vmin + 1e-6

        bins = np.linspace(vmin, vmax, self.num_bins - 1)

        self.feature_bins[name] = bins
        self.feature_mins[name] = vmin
        self.feature_maxs[name] = vmax

    def fit(self, feature_dict):
        for name, values in feature_dict.items():
            self.fit_feature(name, values)

    def tokenize(self, name, values):
        bins = self.feature_bins[name]
        values = np.asarray(values, dtype=np.float32)
        tokens = np.digitize(values, bins, right=False)
        return tokens.astype(np.int64)

    def detokenize(self, name, tokens):
        bins = self.feature_bins[name]
        tokens = np.asarray(tokens, dtype=np.int64)

        edges = np.concatenate((
            [self.feature_mins[name]],
            bins,
            [self.feature_maxs[name]]
        ))
        centers = 0.5 * (edges[:-1] + edges[1:])
        idx = np.clip(tokens, 0, len(centers) - 1)
        return centers[idx].astype(np.float32)

In [101]:
# ==========================================
# 5. PŘÍPRAVA DAT PRO MODEL
# ==========================================
class TrackDataProcessor:
    def __init__(self, config):
        self.window_size = config.window_size
        self.tokenizer = FeatureTokenizer(num_bins=config.default_num_bins)
        self.input_feature_names = ALL_INPUT_FEATURES
        self.target_names = TARGETS
        self.vocab_size = config.default_num_bins

    def build_feature_dict(self, g1, g2, g3):
        feature_dict = {}

        # Základní G1 a G2 příznaky
        for col in G1_FEATURES:
            feature_dict[col] = g1[col].values.astype(np.float32)
        for col in G2_FEATURES:
            feature_dict[col] = g2[col].values.astype(np.float32)

        # D1 a D2 příznaky
        for col in D1_FEATURES:
            feature_dict[col] = g1[col].values.astype(np.float32)
        for col in D2_FEATURES:
            feature_dict[col] = g2[col].values.astype(np.float32)

        # Výpočet DELTA FEATURES
        feature_dict['delta_posun'] = (
            g2['posun [mm]_G2'].values.astype(np.float32) -
            g1['posun [mm]_G1'].values.astype(np.float32)
        )
        feature_dict['delta_zdvih'] = (
            g2['zdvih [mm]_G2'].values.astype(np.float32) -
            g1['zdvih [mm]_G1'].values.astype(np.float32)
        )

        # Snaha vidět trend a odfiltrovat šum
        df_temp = pd.DataFrame({
            'p': feature_dict['posun [mm]_G2'],
            'z': feature_dict['zdvih [mm]_G2']
        })
        feature_dict['posun_G2_roll5'] = df_temp['p'].rolling(5, min_periods=1).mean().values.astype(np.float32)
        feature_dict['zdvih_G2_roll5'] = df_temp['z'].rolling(5, min_periods=1).mean().values.astype(np.float32)

        # Targety a pozice
        for col in TARGETS:
            feature_dict[col] = g3[col].values.astype(np.float32)
        feature_dict['Pos_m'] = g3['Pos_m'].values.astype(np.float32)

        return feature_dict

    def fit_tokenizer(self, g1, g2, g3):
        feature_dict = self.build_feature_dict(g1, g2, g3)
        # Filtrace na všech vstupech i targetech, aby tokenizer znal rozsahy
        fit_dict = {name: feature_dict[name] for name in self.input_feature_names + self.target_names}
        self.tokenizer.fit(fit_dict)

    def prepare(self, g1, g2, g3):
        self.fit_tokenizer(g1, g2, g3)
        feature_dict = self.build_feature_dict(g1, g2, g3)

        # Tokenizace všech vstupů
        tokenized_inputs = {}
        for name in self.input_feature_names:
            tokenized_inputs[name] = self.tokenizer.tokenize(name, feature_dict[name])

        # Tokenizace targetů
        tokenized_targets = {}
        for name in self.target_names:
            tokenized_targets[name] = self.tokenizer.tokenize(name, feature_dict[name])

        X, y, pos = [], [], []
        n = len(g1)
        for i in range(n - self.window_size):
            seq_features = []
            for name in self.input_feature_names:
                seq_features.append(tokenized_inputs[name][i:i+self.window_size])

            seq = np.stack(seq_features, axis=-1)

            # Predikce hodnot G3 na konci okna
            target_t = [
                tokenized_targets['posun [mm]_G3'][i + self.window_size],
                tokenized_targets['zdvih [mm]_G3'][i + self.window_size]
            ]

            X.append(seq)
            y.append(target_t)
            pos.append(feature_dict['Pos_m'][i + self.window_size])

        return torch.LongTensor(np.array(X)), torch.LongTensor(np.array(y)), np.array(pos, dtype=np.float32)

In [102]:
# ==========================================
# 6. MODEL
# ==========================================
class TransformerBlock(nn.Module):
    def __init__(self, embed_size, n_heads, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_size)
        self.attn = nn.MultiheadAttention(
            embed_size,
            n_heads,
            dropout=dropout,
            batch_first=True
        )
        self.ln2 = nn.LayerNorm(embed_size)
        self.mlp = nn.Sequential(
            nn.Linear(embed_size, 4 * embed_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * embed_size, embed_size),
            nn.Dropout(dropout)
        )

    def forward(self, x, attn_mask=None):
        # Klasická Transformer architektura s reziduálními spoji
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=attn_mask)
        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x

class TrackGPTMulti(nn.Module):
    def __init__(self, vocab_size, num_features, window_size, config):
        super().__init__()
        self.vocab_size = vocab_size
        self.num_features = num_features
        self.window_size = window_size
        self.embed_size = config.embed_size

        # Embeddings
        self.token_emb = nn.Embedding(vocab_size, config.embed_size)
        self.feature_emb = nn.Embedding(num_features, config.embed_size)
        self.pos_emb = nn.Parameter(torch.zeros(1, window_size, config.embed_size))

        # Projekce: Sloučení informací ze všech příznaků v daném čase
        self.feature_proj = nn.Linear(num_features * config.embed_size, config.embed_size)

        self.blocks = nn.ModuleList([
            TransformerBlock(config.embed_size, config.n_heads, config.dropout)
            for _ in range(config.n_layers)
        ])

        self.ln_f = nn.LayerNorm(config.embed_size)

        # Klasifikační hlavy pro biny (posun a zdvih)
        self.head_posun = nn.Linear(config.embed_size, vocab_size)
        self.head_zdvih = nn.Linear(config.embed_size, vocab_size)

    def make_causal_mask(self, T, device):
        # Maska zabraňující nahlížení do budoucnosti v rámci okna
        return torch.triu(torch.ones(T, T, device=device), diagonal=1).bool()

    def forward(self, x):
        B, T, F = x.shape
        device = x.device

        # 1. Pro každý příznak (F) získání embedding a ID příznaku
        feature_vectors = []
        for f in range(F):
            tok = self.token_emb(x[:, :, f])
            feat_id = torch.full((B, T), f, device=device, dtype=torch.long)
            feat = self.feature_emb(feat_id)
            feature_vectors.append(tok + feat)

        # 2. Sloučení příznaků
        h = torch.cat(feature_vectors, dim=-1)
        h = self.feature_proj(h) + self.pos_emb[:, :T, :]

        # 3. Průchod transformer bloky
        attn_mask = self.make_causal_mask(T, device)
        for block in self.blocks:
            h = block(h, attn_mask=attn_mask)

        # 4. Výstup pouze z posledního časového kroku okna
        h = self.ln_f(h)
        last = h[:, -1, :]

        logits_posun = self.head_posun(last)
        logits_zdvih = self.head_zdvih(last)

        return logits_posun, logits_zdvih

In [ ]:
# ==========================================
# 7. DATASET A SPLIT
# ==========================================
processor = TrackDataProcessor(config)
X, y, pos_m = processor.prepare(df_g1, df_g2, df_g3)

split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
pos_train, pos_val = pos_m[:split], pos_m[split:]

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=config.batch_size,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=config.batch_size,
    shuffle=False
)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# ==========================================
# 8. TRÉNINK
# ==========================================
model = TrackGPTMulti(
    vocab_size=processor.vocab_size,
    num_features=len(processor.input_feature_names),
    window_size=config.window_size,
    config=config
).to(config.device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.lr,
    weight_decay=config.weight_decay
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

history = {
    "train_loss": [],
    "val_mae_posun": [],
    "val_mae_zdvih": []
}

best_score = float("inf")
best_state = None
best_epoch = -1

print(f"\nStart tréninku na: {config.device}")

for epoch in range(config.epochs):
    model.train()
    train_loss_sum = 0.0

    for xb, yb in train_loader:
        xb = xb.to(config.device)
        yb = yb.to(config.device)

        optimizer.zero_grad()

        logits_posun, logits_zdvih = model(xb)

        loss_posun = criterion(logits_posun, yb[:, 0])
        loss_zdvih = criterion(logits_zdvih, yb[:, 1])
        loss = loss_posun + loss_zdvih

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item()

    model.eval()
    pred_posun_all, pred_zdvih_all = [], []
    real_posun_all, real_zdvih_all = [], []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(config.device)

            logits_posun, logits_zdvih = model(xb)

            pred_posun = torch.argmax(logits_posun, dim=1).cpu().numpy()
            pred_zdvih = torch.argmax(logits_zdvih, dim=1).cpu().numpy()

            real_posun = yb[:, 0].cpu().numpy()
            real_zdvih = yb[:, 1].cpu().numpy()

            pred_posun_all.extend(pred_posun)
            pred_zdvih_all.extend(pred_zdvih)
            real_posun_all.extend(real_posun)
            real_zdvih_all.extend(real_zdvih)

    pred_posun_mm = processor.tokenizer.detokenize('posun [mm]_G3', np.array(pred_posun_all))
    pred_zdvih_mm = processor.tokenizer.detokenize('zdvih [mm]_G3', np.array(pred_zdvih_all))

    real_posun_mm = processor.tokenizer.detokenize('posun [mm]_G3', np.array(real_posun_all))
    real_zdvih_mm = processor.tokenizer.detokenize('zdvih [mm]_G3', np.array(real_zdvih_all))

    mae_posun = np.mean(np.abs(pred_posun_mm - real_posun_mm))
    mae_zdvih = np.mean(np.abs(pred_zdvih_mm - real_zdvih_mm))

    history["train_loss"].append(train_loss_sum / len(train_loader))
    history["val_mae_posun"].append(mae_posun)
    history["val_mae_zdvih"].append(mae_zdvih)

    score = mae_posun + mae_zdvih
    if score < best_score:
        best_score = score
        best_epoch = epoch + 1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss: {history['train_loss'][-1]:.4f} | "
            f"MAE posun: {mae_posun:.3f} mm | "
            f"MAE zdvih: {mae_zdvih:.3f} mm"
        )

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nNačten nejlepší model z epochy {best_epoch} | score={best_score:.3f}")

In [ ]:
# ==========================================
# 9. FINÁLNÍ VYHODNOCENÍ A GRAFY
# ==========================================
model.eval()

with torch.no_grad():
    logits_posun, logits_zdvih = model(X_val.to(config.device))

    pred_posun = torch.argmax(logits_posun, dim=1).cpu().numpy()
    pred_zdvih = torch.argmax(logits_zdvih, dim=1).cpu().numpy()

    real_posun = y_val[:, 0].cpu().numpy()
    real_zdvih = y_val[:, 1].cpu().numpy()

pred_posun_mm = processor.tokenizer.detokenize('posun [mm]_G3', pred_posun)
pred_zdvih_mm = processor.tokenizer.detokenize('zdvih [mm]_G3', pred_zdvih)

real_posun_mm = processor.tokenizer.detokenize('posun [mm]_G3', real_posun)
real_zdvih_mm = processor.tokenizer.detokenize('zdvih [mm]_G3', real_zdvih)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

axes[0, 0].plot(history["train_loss"])
axes[0, 0].set_title("Train loss")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history["val_mae_posun"], label="MAE posun")
axes[0, 1].plot(history["val_mae_zdvih"], label="MAE zdvih")
axes[0, 1].set_title("Validační MAE [mm]")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(pos_val, real_posun_mm, label="Realita G3 posun", alpha=0.7)
axes[1, 0].plot(pos_val, pred_posun_mm, label="Predikce G3 posun", linestyle="--")
axes[1, 0].set_title("G3 posun: realita vs predikce")
axes[1, 0].set_xlabel("Pos_m")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(pos_val, real_zdvih_mm, label="Realita G3 zdvih", alpha=0.7)
axes[1, 1].plot(pos_val, pred_zdvih_mm, label="Predikce G3 zdvih", linestyle="--")
axes[1, 1].set_title("G3 zdvih: realita vs predikce")
axes[1, 1].set_xlabel("Pos_m")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinální MAE posun: {np.mean(np.abs(pred_posun_mm - real_posun_mm)):.3f} mm")
print(f"Finální MAE zdvih: {np.mean(np.abs(pred_zdvih_mm - real_zdvih_mm)):.3f} mm")